In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, math

df = pd.read_parquet("/content/drive/MyDrive/esci/processed/rerank_pairs_v2.3.parquet")
print(df.shape, df["part"].value_counts().to_dict())

def _clean(x):
    if x is None or (isinstance(x, float) and math.isnan(x)): return ""
    return " ".join(str(x).split())          # removes extra spaces and new lines

def product_to_text(p: dict, style: str = "full", bullet_chars: int = 200) -> str:
    """ONLY place that turns a product into model text.
    p uses generic keys: title, brand, color, bullets  (Member 2 fields go here later)."""
    title, brand, color = _clean(p.get("title")), _clean(p.get("brand")), _clean(p.get("color"))
    if style == "title":          parts = [title]
    elif style == "title_brand":  parts = [title, brand]
    elif style == "full":         parts = [title, brand, color]
    elif style == "full_bullets": parts = [title, brand, color, _clean(p.get("bullets"))[:bullet_chars]]
    else: raise ValueError(style)
    return " | ".join(x for x in parts if x)

def esci_row_to_product(r) -> dict:
    """Maps ESCI column names to the generic keys."""
    return {"title": r["product_title"], "brand": r["product_brand"],
            "color": r["product_color"], "bullets": r["product_bullet_point"]}

# test on 3 rows
STYLES = ["title", "title_brand", "full", "full_bullets"]
for _, r in df.sample(3, random_state=1).iterrows():
    print("\nQUERY:", r["query"], "| label:", r["esci_label"])
    for s in STYLES:
        print(f"  {s:13s} ->", product_to_text(esci_row_to_product(r), s)[:160])

Mounted at /content/drive
(100740, 16) {'train': 65852, 'test': 23147, 'val': 11741}

QUERY: car wifi router | label: S
  title         -> RAVPower FileHub, Travel Router AC750, Wireless SD Card Reader, Connect Portable SSD Hard Drive to iPhone iPad Tablet Smart Phone Laptop for Photo Backup, Data 
  title_brand   -> RAVPower FileHub, Travel Router AC750, Wireless SD Card Reader, Connect Portable SSD Hard Drive to iPhone iPad Tablet Smart Phone Laptop for Photo Backup, Data 
  full          -> RAVPower FileHub, Travel Router AC750, Wireless SD Card Reader, Connect Portable SSD Hard Drive to iPhone iPad Tablet Smart Phone Laptop for Photo Backup, Data 
  full_bullets  -> RAVPower FileHub, Travel Router AC750, Wireless SD Card Reader, Connect Portable SSD Hard Drive to iPhone iPad Tablet Smart Phone Laptop for Photo Backup, Data 

QUERY: usb c charger | label: E
  title         -> Anker USB C Charger 20W, 511 Charger ( Nano ), PIQ 3.0 Durable Compact Fast Charger, Anker Nano for iPhone 1

In [2]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("cross-encoder/ms-marco-MiniLM-L-6-v2")
sample = df.sample(3000, random_state=0)
for s in STYLES:
    lens = [len(tok(q, product_to_text(esci_row_to_product(r), s))["input_ids"])
            for q, (_, r) in zip(sample["query"], sample.iterrows())]
    lens = np.array(lens)
    print(f"{s:13s} median {int(np.median(lens)):4d} | 95% {int(np.percentile(lens,95)):4d} | "
          f"over 128: {(lens>128).mean():.1%} | over 256: {(lens>256).mean():.1%}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

title         median   46 | 95%   76 | over 128: 0.0% | over 256: 0.0%
title_brand   median   49 | 95%   80 | over 128: 0.0% | over 256: 0.0%
full          median   51 | 95%   83 | over 128: 0.0% | over 256: 0.0%
full_bullets  median   96 | 95%  143 | over 128: 10.9% | over 256: 0.0%


In [3]:
def ndcg_at_k(ranked_gains, all_gains, k=10):
    g = np.asarray(ranked_gains, float)[:k]
    disc = 1.0 / np.log2(np.arange(2, len(g) + 2))
    dcg = float((g * disc).sum())
    ideal = np.sort(np.asarray(all_gains, float))[::-1][:k]
    idcg = float((ideal * (1.0 / np.log2(np.arange(2, len(ideal) + 2)))).sum())
    return dcg / idcg if idcg > 0 else 0.0

def mrr(ranked_gains):
    for i, g in enumerate(ranked_gains, 1):
        if g >= 1.0: return 1.0 / i          # first Exact product
    return 0.0

def precision_at_k(ranked_gains, k=5):
    return float((np.asarray(ranked_gains[:k]) >= 1.0).sum()) / k

def evaluate(d, score_col="score", seed=0):
    """d: rows of one part (val/test) with query_id, gain and a score column.
    Returns (mean metrics dict, per-query table)."""
    d = d.copy()
    d["_tie"] = np.random.default_rng(seed).random(len(d))      # fixed random tie-break
    d = d.sort_values(["query_id", score_col, "_tie"], ascending=[True, False, True])
    rows = []
    for qid, grp in d.groupby("query_id", sort=False):
        g = grp["gain"].tolist()
        rows.append({"query_id": qid, "ndcg@10": ndcg_at_k(g, g, 10),
                     "mrr": mrr(g), "p@5": precision_at_k(g, 5)})
    per_q = pd.DataFrame(rows)
    return per_q[["ndcg@10", "mrr", "p@5"]].mean().round(4).to_dict(), per_q

# ---- test 1: a tiny example worked out by hand
g = [0.0, 1.0, 0.1]                     # ranked: I, E, S
print("ndcg:", round(ndcg_at_k(g, g), 4), "(expect 0.6405)",
      "| mrr:", mrr(g), "(expect 0.5)", "| p@5:", precision_at_k(g), "(expect 0.2)")

# ---- test 2: on the real validation set
val = df[df["part"] == "val"].copy()
val["score"] = val["gain"]                               # perfect order
print("perfect  :", evaluate(val)[0])
val["score"] = -val["gain"]                              # worst order
print("worst    :", evaluate(val)[0])
val["score"] = np.random.default_rng(1).random(len(val)) # random order
print("random   :", evaluate(val)[0])
print("queries evaluated:", val["query_id"].nunique())   # 656

ndcg: 0.6405 (expect 0.6405) | mrr: 0.5 (expect 0.5) | p@5: 0.2 (expect 0.2)
perfect  : {'ndcg@10': 1.0, 'mrr': 1.0, 'p@5': 0.9116}
worst    : {'ndcg@10': 0.5579, 'mrr': 0.4353, 'p@5': 0.4454}
random   : {'ndcg@10': 0.7682, 'mrr': 0.7759, 'p@5': 0.6838}
queries evaluated: 656
